In [3]:
# Jupyter 批量绘制 UMAP（散点仅栅格化，轴/文字保持矢量）
# 说明：
# - 批量读取指定数据集列表对应的 h5ad 路径 f"D:/Data/{dataset}_cleaned.h5ad"
# - 使用给定的 plot_umap_without_borders() 风格绘图
# - 在导出为 PDF/SVG 时仅将散点层设为 rasterized，轴与文字仍为矢量，便于在 AI(Adobe Illustrator) 中编辑
# - 若对象缺少 UMAP，可自动计算 PCA→neighbors→UMAP（可通过开关控制）
# - 默认输出 PDF 到 ./umap_out/

import os
import gc
import warnings
from pathlib import Path

import numpy as np
import scanpy as sc
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt

# 让 PDF/SVG 内文字可被 AI 编辑
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'

warnings.filterwarnings("ignore", category=UserWarning)

# ========================= 可配置参数 =========================
# 你可以在这里替换为自己的列表；此处给出示例：
datasetlist = ['M-MG','R-MG','S-MG','S-AG','R-AG','R-CG']

# h5ad 路径模板（根据你的目录结构修改）
H5AD_TEMPLATE = r"D:/Data/{dataset}_cleaned.h5ad"

# 着色列（.obs 中的列名）
COLOR_COL = 'newcelltype'

# 若缺少 UMAP，是否自动计算
RECOMPUTE_UMAP_IF_MISSING = True
N_NEIGHBORS = 15
N_PCS = 50
RANDOM_STATE = 0

# 绘图尺寸与样式
FIG_WIDTH, FIG_HEIGHT = 5.5, 4.5
POINT_SIZE = 3.0

# 导出设置
OUT_DIR = Path('./umap_out')
OUT_FMT = 'pdf'  # 可选：'pdf'/'svg'/'png'
DPI = 300        # 对 png 有效；pdf/svg 为矢量容器 + 栅格散点层
SHOW_IN_NOTEBOOK = False  # 若为 True，将在输出前显示图像（大量图时建议 False）
SKIP_IF_MISSING_COLOR = True  # 若缺少 COLOR_COL：True=跳过；False=用统一颜色继续

# =============================================================

def plot_umap_without_borders(ea, dataset, color="newcelltype", size=3):
    """
    绘制没有边框、标题的 UMAP 图，在左下角添加一个坐标轴。
    返回 Figure 对象。
    """
    fig = sc.pl.umap(
        ea,
        color=color,   # 可为 None；则为统一配色
        size=size,
        show=False,
        return_fig=True
    )

    ax = fig.axes[0]

    # 删除标题
    ax.set_title("")

    # 删除坐标轴刻度和标签
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    # 删除边框
    for spine in ax.spines.values():
        spine.set_visible(False)

    # 左下角小坐标箭头
    arrow_length = 0.2  # 坐标轴长度（相对 axes fraction）
    arrow_x_start = 0.05
    arrow_y_start = 0.05

    # UMAP1
    ax.annotate(
        "", xy=(arrow_length, 0), xytext=(0, 0),
        arrowprops=dict(facecolor="black", shrink=0, width=1, headwidth=5),
        xycoords="axes fraction", textcoords="axes fraction",
        annotation_clip=False
    )

    # UMAP2
    ax.annotate(
        "", xy=(0, arrow_length), xytext=(0, 0),
        arrowprops=dict(facecolor="black", shrink=0, width=1, headwidth=5),
        xycoords="axes fraction", textcoords="axes fraction",
        annotation_clip=False
    )

    # 标签
    ax.text(
        arrow_x_start + 0.2, arrow_y_start, "UMAP1",
        fontsize=10, ha="center", va="center", transform=ax.transAxes
    )
    ax.text(
        arrow_x_start, arrow_y_start + 0.2, "UMAP2",
        fontsize=10, ha="center", va="center", transform=ax.transAxes
    )

    return fig


def rasterize_scatter_layers(fig, rasterize_legends=True):
    """仅将散点层(PathCollection)标记为 rasterized，轴与文字保持矢量；
    对图例在 Matplotlib 不同版本的属性差异做兼容处理。
    """
    from matplotlib.collections import PathCollection
    from matplotlib.lines import Line2D

    for ax in fig.axes:
        # 主图中的散点集合
        for coll in getattr(ax, "collections", []):
            try:
                coll.set_rasterized(True)
                if hasattr(coll, "set_linewidths"):
                    coll.set_linewidths(0)
            except Exception:
                pass

        # 图例：不同版本可能无 legendHandles；做多分支兼容
        if rasterize_legends:
            leg = getattr(ax, "legend_", None)
            if leg is not None:
                handles = []
                # 1) 常见属性名
                for attr in ("legendHandles", "legend_handles"):
                    if hasattr(leg, attr):
                        try:
                            handles = list(getattr(leg, attr))
                            break
                        except Exception:
                            handles = []
                # 2) 若上述失败，遍历图例内部的可见图元（避免栅格化文字）
                if not handles:
                    try:
                        handles = [a for a in leg.findobj((PathCollection, Line2D))]
                    except Exception:
                        handles = []
                # 应用栅格化
                for h in handles:
                    try:
                        h.set_rasterized(True)
                        if isinstance(h, PathCollection) and hasattr(h, "set_linewidths"):
                            h.set_linewidths(0)
                    except Exception:
                        pass

    return fig


def ensure_umap(adata, n_neighbors=15, n_pcs=50, random_state=0):
    """确保 .obsm['X_umap'] 存在；若不存在则计算 PCA/邻居/UMAP。"""
    if 'X_umap' in adata.obsm_keys():
        return adata
    if 'X_pca' not in adata.obsm_keys():
        sc.pp.pca(adata, n_comps=min(n_pcs, adata.n_vars), svd_solver='arpack')
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=min(n_pcs, adata.obsm['X_pca'].shape[1]))
    sc.tl.umap(adata, random_state=random_state)
    return adata


# ========================= 主流程（Jupyter 单元格直接运行） =========================
OUT_DIR.mkdir(parents=True, exist_ok=True)
sc.set_figure_params(figsize=(FIG_WIDTH, FIG_HEIGHT))

for dataset in datasetlist:
    h5ad_path = Path(H5AD_TEMPLATE.format(dataset=dataset))
    print(f"[INFO] {dataset}: 读取 {h5ad_path}")
    if not h5ad_path.exists():
        print(f"[WARN] {dataset}: 文件不存在，跳过。")
        continue

    try:
        adata = ad.read_h5ad(h5ad_path)
    except Exception as e:
        print(f"[WARN] {dataset}: 读取失败，跳过。原因: {e}")
        continue

    # 确保 UMAP
    if 'X_umap' not in adata.obsm_keys():
        if RECOMPUTE_UMAP_IF_MISSING:
            print(f"[INFO] {dataset}: 未检测到 UMAP，开始计算 (n_neighbors={N_NEIGHBORS}, n_pcs={N_PCS})")
            try:
                ensure_umap(adata, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS, random_state=RANDOM_STATE)
            except Exception as e:
                print(f"[WARN] {dataset}: 计算 UMAP 失败，跳过。原因: {e}")
                del adata
                gc.collect()
                continue
        else:
            print(f"[WARN] {dataset}: 未检测到 UMAP，且未启用自动计算，跳过。")
            del adata
            gc.collect()
            continue

    # 颜色列处理
    color_arg = COLOR_COL
    if COLOR_COL not in adata.obs.columns:
        msg = f"[WARN] {dataset}: .obs 缺少列 '{COLOR_COL}'"
        if SKIP_IF_MISSING_COLOR:
            print(msg + "，将使用统一颜色继续绘制。")
            color_arg = None
        else:
            print(msg + "，跳过该数据集。")
            del adata
            gc.collect()
            continue

    # 绘图
    try:
        fig = plot_umap_without_borders(adata, dataset, color=color_arg, size=POINT_SIZE)
        rasterize_scatter_layers(fig, rasterize_legends=True)

        out_file = OUT_DIR / f"{dataset}_umap_{COLOR_COL}.{OUT_FMT}"
        if OUT_FMT.lower() in {"pdf", "svg"}:
            fig.savefig(out_file, dpi=DPI, bbox_inches='tight')
        else:
            fig.savefig(out_file, dpi=DPI, bbox_inches='tight', transparent=True)

        if SHOW_IN_NOTEBOOK:
            from IPython.display import display
            display(fig)

        plt.close(fig)
        print(f"[OK] 已保存：{out_file}")
    except Exception as e:
        print(f"[WARN] {dataset}: 绘制/保存失败。原因: {e}")
    finally:
        del adata
        gc.collect()

print("[DONE] 全部完成。")


[INFO] M-MG: 读取 D:\Data\M-MG_cleaned.h5ad
[OK] 已保存：umap_out\M-MG_umap_newcelltype.pdf
[INFO] R-MG: 读取 D:\Data\R-MG_cleaned.h5ad
[OK] 已保存：umap_out\R-MG_umap_newcelltype.pdf
[INFO] S-MG: 读取 D:\Data\S-MG_cleaned.h5ad
[OK] 已保存：umap_out\S-MG_umap_newcelltype.pdf
[INFO] S-AG: 读取 D:\Data\S-AG_cleaned.h5ad
[OK] 已保存：umap_out\S-AG_umap_newcelltype.pdf
[INFO] R-AG: 读取 D:\Data\R-AG_cleaned.h5ad
[OK] 已保存：umap_out\R-AG_umap_newcelltype.pdf
[INFO] R-CG: 读取 D:\Data\R-CG_cleaned.h5ad
[OK] 已保存：umap_out\R-CG_umap_newcelltype.pdf
[DONE] 全部完成。
